# End-to-end walkthrough — Module A pipeline

**Objective:** Trace a single **entity** row from synthetic generation through every pipeline stage:  
generation → flaw injection → cleaning → feature engineering → segmentation → propensity scoring.

All RNG seeds fixed to **42** for full reproducibility.

| Stage | Module | Output shown |
|-------|--------|--------------|
| 1. Generate | `data.generator` | `entity_id`, age, department, `has_cedula` |
| 2. Inject flaws | `data.raw_injector` | duplicate count, null delta |
| 3. Clean | `data.cleaner` | 14-step QA, clean row |
| 4. Features | `features.*` | behavioral + demographic + reachability |
| 5. Segment | `models.segmentation` | `segment_label`, silhouette |
| 6. Propensity | `models.propensity` | `participation_propensity`, AUC-ROC |

In [1]:
# --- Setup ---------------------------------------------------------------
# Resolve the repo root robustly. nbconvert executes a notebook with cwd set
# to the notebook's own directory by default, so we walk upward from cwd
# looking for the repo's single top-level `pyproject.toml` rather than
# assuming any particular invocation directory.
#
# Config/anchor loading mirrors `population_segmentation.pipeline.__main__`
# (the CLI entry point): both files are plain YAML, loaded with
# `yaml.safe_load` — there is no separate loader function in the pipeline
# package to import, so this reproduces that inline logic exactly rather
# than duplicating a different code path.
#
# Sample size: `generation.yaml` ships with `sample_size: 100000`. Running
# the full generate -> inject -> clean -> features -> segment -> propensity
# chain at that scale takes well over 10 minutes end-to-end. We override it
# to 10,000 here -- the same "minutes, not hours" dev sample size `make
# pipeline-dev` defaults to (see Makefile) -- purely so this walkthrough
# executes quickly. This does not weaken the determinism guarantee below:
# SEED=42 is fixed at every stage regardless of sample size, so re-running
# this notebook at SAMPLE_SIZE=10_000 reproduces identical outputs.
from pathlib import Path
from typing import Any

import yaml

SEED = 42
SAMPLE_SIZE = 10_000  # override; config default (100,000) is too slow for a walkthrough


def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError(f"could not locate repo root (pyproject.toml) above {start}")


ROOT = _find_repo_root(Path.cwd())

config_path = ROOT / "module_a_population_segmentation/config/generation.yaml"
anchors_path = ROOT / "module_a_population_segmentation/config/calibration_anchors.yaml"

with open(config_path, encoding="utf-8") as f:
    config: dict[str, Any] = yaml.safe_load(f)
with open(anchors_path, encoding="utf-8") as f:
    anchors: dict[str, Any] = yaml.safe_load(f)

original_sample_size = config["sample_size"]
config["sample_size"] = SAMPLE_SIZE

print(f"ROOT: {ROOT}")
print(
    f"sample_size: {config['sample_size']:,} "
    f"(config default is {original_sample_size:,}; overridden here for walkthrough speed)"
)


ROOT: C:\Users\Benutzer1\Dev\decision-analytics-reconstruction
sample_size: 10,000 (config default is 100,000; overridden here for walkthrough speed)


## Determinism guarantee

All random operations in this notebook are seeded with **SEED=42** (generation, flaw injection, cleaning, segmentation k-means, propensity model, stratification). **Re-running this notebook cell-by-cell produces identical outputs** — parquet rows match, metrics match to floating-point precision (< 1 ULP). This seeding is enforced at the pipeline entry point (`population_segmentation.pipeline.main --seed 42`) and in every stochastic function call. If outputs differ, check Python version (3.11.x required), dependency lock file (`poetry lock --no-update`), and CPU architecture.

## Step 1 — Generate synthetic population

`generate_population` creates N entities from the YAML distribution spec.  
Values mirror 2018 census department weights — no real PII.

In [2]:
from population_segmentation.data.generator import generate_population

raw = generate_population(config, seed=42)
print(f"Generated {len(raw):,} entities, {raw.shape[1]} columns")

ENTITY_IDX = 0
entity_id = int(raw["entity_id"].iloc[ENTITY_IDX])
print(f"\n--- Entity #{entity_id} (raw, before any flaws) ---")
display(raw[raw["entity_id"] == entity_id][["entity_id", "age_on_event_date", "gender", "department", "rural_flag", "language_census_bucket"]].T)

Generated 10,000 entities, 20 columns

--- Entity #1 (raw, before any flaws) ---


,0
entity_id,1
age_on_event_date,32
gender,F
department,Paraguari
rural_flag,True
language_census_bucket,spanish_only


## Step 2 — Inject realistic data flaws

`inject_flaws` simulates survey-quality issues: duplicate rows, null fields, cedula format errors.  
Flaw rates are controlled by `config['flaw_rates']` in `generation.yaml`.

In [3]:
from population_segmentation.data.raw_injector import inject_flaws

raw_dirty = inject_flaws(raw, config, seed=42)

print(f"Rows:  {len(raw):,} → {len(raw_dirty):,}  (+{len(raw_dirty) - len(raw)} duplicate rows injected)")
print(f"Nulls: {raw.isnull().sum().sum():,} → {raw_dirty.isnull().sum().sum():,}")

dirty_rows = raw_dirty[raw_dirty["entity_id"] == entity_id]
print(f"\n--- Entity #{entity_id} appears {len(dirty_rows)} time(s) in dirty frame ---")
display(dirty_rows[["entity_id", "age_on_event_date", "gender", "department", "cedula"]].T)

Rows:  10,000 → 10,120  (+120 duplicate rows injected)
Nulls: 0 → 3,383

--- Entity #1 appears 1 time(s) in dirty frame ---


,0
entity_id,1
age_on_event_date,32
gender,F
department,Paraguari
cedula,18032585


## Step 3 — 14-step cleaning pipeline

`clean_population` runs 14 sequential QA gates: dedup, null imputation, cedula normalisation,  
age clamping, department validation, outlier flags, and more.  
A QA report is written to disk at each run.

In [4]:
import tempfile
from pathlib import Path

from population_segmentation.data.cleaner import clean_population

qa_dir = Path(tempfile.mkdtemp())
clean_df = clean_population(raw_dirty, config, qa_report_dir=qa_dir, seed=42)

print(f"Rows after cleaning: {len(clean_df):,}  (removed {len(raw_dirty) - len(clean_df):,} rows)")
print(f"Null cells remaining: {clean_df.isnull().sum().sum()}")

clean_row = clean_df[clean_df["entity_id"] == entity_id]
if len(clean_row):
    print(f"\n--- Entity #{entity_id} after cleaning ---")
    display(clean_row[["entity_id", "age_on_event_date", "gender", "department", "cedula", "cedula_invalid", "rural_flag"]].T)
else:
    print(f"Entity #{entity_id} removed by a QA gate — using next available entity.")
    entity_id = int(clean_df["entity_id"].iloc[0])
    display(clean_df[clean_df["entity_id"] == entity_id][["entity_id", "age_on_event_date", "gender", "department", "cedula"]].T)

Rows after cleaning: 10,000  (removed 120 rows)
Null cells remaining: 2571

--- Entity #1 after cleaning ---


,0
entity_id,1
age_on_event_date,31
gender,F
department,Paraguari
cedula,18032585
cedula_invalid,False
rural_flag,True


## Step 4 — Feature engineering

Three layers stacked in sequence:
- **Demographic** — age group, dependency ratio, education index  
- **Behavioral** — structural dependency flag, jopará index  
- **Reachability** — TV/radio/WhatsApp penetration, internet access, primary reach channel

In [5]:
from population_segmentation.features.behavioral import build_behavioral_features
from population_segmentation.features.demographic import build_demographic_features
from population_segmentation.features.reachability import build_reachability_features

feat_df = build_reachability_features(
    build_behavioral_features(build_demographic_features(clean_df))
)

new_cols = [c for c in feat_df.columns if c not in clean_df.columns]
print(f"Feature columns added: {len(new_cols)}")
print(f"  {new_cols}")

print(f"\n--- Entity #{entity_id} feature vector ---")
display(feat_df[feat_df["entity_id"] == entity_id][new_cols].T)

Feature columns added: 31
  ['age_bin', 'age_bin_encoded', 'gender_encoded', 'gender_is_M', 'gender_is_F', 'gender_is_unknown', 'youth_flag', 'senior_flag', 'chaco_flag', 'department_region', 'metro_flag', 'preference_proxy_encoded', 'preference_proxy_is_A', 'preference_proxy_is_B', 'preference_proxy_is_other', 'preference_proxy_is_none', 'structural_dependency_encoded', 'nbi_stress_prior_scaled', 'language_jopara_encoded', 'language_guarani_flag', 'reachability_digital', 'reachability_broadcast_tv', 'reachability_broadcast_radio', 'reachability_facebook_ads', 'reachability_instagram_ads', 'reachability_google_ads', 'reachability_linkedin_ads', 'reachability_index', 'reachability_tier', 'urban_digital_compound', 'rural_offline_compound']

--- Entity #1 feature vector ---


,0
age_bin,25_34
age_bin_encoded,1
gender_encoded,0.0
gender_is_M,0.0
gender_is_F,1.0
gender_is_unknown,0.0
youth_flag,False
senior_flag,False
chaco_flag,False
department_region,ORIENTAL


## Step 5 — Segmentation (k-means, k=6)

`build_segmentation_frame` runs DBSCAN noise filtering then k-means on PCA-reduced features.  
Returns human-readable segment labels (e.g. `high_reach_urban`, `rural_low_contact`)  
plus scalar diagnostics: silhouette score, bootstrap ARI, noise rate.

In [6]:
from population_segmentation.models.segmentation import build_segmentation_frame

labels_df, seg_metrics = build_segmentation_frame(feat_df, k=6, random_state=42)

print("Segmentation diagnostics:")
for k, v in seg_metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

print("\nSegment distribution:")
display(labels_df["segment_label"].value_counts().to_frame("count"))

seg_row = labels_df[labels_df["entity_id"] == entity_id]
print(f"\n--- Entity #{entity_id} segment assignment ---")
display(seg_row[["entity_id", "segment_label", "segment_id", "dbscan_noise_flag"]].T)

Segmentation diagnostics:
  silhouette: 0.2600
  bootstrap_ari: 0.5323
  noise_rate: 0.0020
  davies_bouldin: 1.4180
  calinski_harabasz: 2686.6530

Segment distribution:


,count
segment_label,
rural_low_propensity,2461
rural_committed,1940
urban_high_volatility,1601
structurally_dependent_bloc,1503
committed_opposition,1379
youth_volatile,1116



--- Entity #1 segment assignment ---


,0
entity_id,1
segment_label,structurally_dependent_bloc
segment_id,0
dbscan_noise_flag,False


## Step 6 — Participation propensity (logistic regression)

`PropensityModel` trains a stratified logistic regression, then rakes department-level  
scores against census calibration anchors (`calibration_anchors.yaml`).  
Output: `participation_propensity` ∈ [0, 1].

In [7]:
from typing import Any, cast

import yaml

from population_segmentation.models.propensity import PropensityModel

model_params_path = ROOT / "module_a_population_segmentation/config/model_params.yaml"
with open(model_params_path, encoding="utf-8") as f:
    model_params: dict[str, Any] = yaml.safe_load(f)
stratify_by = tuple(model_params["propensity"]["stratify_by"])

merged = feat_df.reset_index(drop=True).copy()
merged["segment_label"] = labels_df["segment_label"].to_numpy()
merged["segment_id"] = labels_df["segment_id"].to_numpy()
merged["dbscan_noise_flag"] = labels_df["dbscan_noise_flag"].to_numpy()

prop_raw = PropensityModel(random_state=42, stratify_by=stratify_by).fit_predict(merged, anchors)
prop_out = cast(dict[str, Any], prop_raw)
prop_metrics = prop_out["metrics"]

print(f"Model evaluation — AUC-ROC: {prop_metrics['auc_roc']:.4f}  Brier score: {prop_metrics['brier_score']:.4f}")

entity_idx = merged[merged["entity_id"] == entity_id].index[0]
propensity = float(prop_out["predictions"].iloc[entity_idx])
logit = float(prop_out["raw_logit_score"].iloc[entity_idx])
rake = float(prop_out["department_rake_multiplier"].iloc[entity_idx])
dept = merged.loc[entity_idx, "department"]
seg_label = merged.loc[entity_idx, "segment_label"]

print(f"\n--- Entity #{entity_id} final scores ---")
print(f"  department:               {dept}")
print(f"  segment:                  {seg_label}")
print(f"  raw logit score:          {logit:.4f}")
print(f"  department rake mult:     {rake:.4f}")
print(f"  participation_propensity: {propensity:.4f}")

Model evaluation — AUC-ROC: 0.8972  Brier score: 0.1162

--- Entity #1 final scores ---
  department:               Paraguari
  segment:                  structurally_dependent_bloc
  raw logit score:          -0.9641
  department rake mult:     2.1984
  participation_propensity: 0.6580


## Summary — entity journey

| Stage | Key output |
|-------|------------|
| Raw generation | Census-calibrated attributes (no PII) |
| Flaw injection | Duplicates + nulls mimicking field survey quality |
| Cleaning | 14 QA gates — dedup, imputation, format normalisation |
| Features | Behavioral + reachability scores appended |
| Segmentation | `segment_label` from k-means (k=6) on PCA features |
| Propensity | `participation_propensity` ∈ [0,1] with department rake |

All seeds fixed to 42 — re-running produces identical outputs.